# Antminer T9+ / S9 Devboard Diagnostics
This notebook validates the core hardware subsystems of your **Bitmain Antminer T9+ (XC7Z010)** devboard:
- Linux Kernel & SoC dual-core ARM Cortex-A9
- 1 GB DDR3 Memory
- Ethernet Networking & IP Addressing
- Internal Zynq On-Chip Temperature & Power Rails (via XADC IIO driver)
- SD Card Filesystem Storage



In [ ]:
import os
import subprocess
import glob

print("==================================================")
print(" 1. LINUX KERNEL & SYSTEM IDENTITY")
print("==================================================")
with open("/proc/version", "r") as f:
    print(f.read().strip())

with open("/etc/os-release", "r") as f:
    for line in f:
        if line.startswith("PRETTY_NAME="):
            print("OS:", line.split("=")[1].strip().strip('"'))



In [ ]:
print("==================================================")
print(" 2. MEMORY & CPU STATUS")
print("==================================================")
# Memory
with open("/proc/meminfo", "r") as f:
    for line in f:
        if any(k in line for k in ["MemTotal:", "MemFree:", "MemAvailable:"]):
            print(line.strip())

# CPU Cores
cores = 0
with open("/proc/cpuinfo", "r") as f:
    for line in f:
        if line.startswith("processor"):
            cores += 1
print(f"Active CPU Cores: {cores}x ARM Cortex-A9")



In [ ]:
print("==================================================")
print(" 3. NETWORKING STATUS")
print("==================================================")
try:
    res = subprocess.run(["ip", "-br", "addr", "show", "eth0"], capture_output=True, text=True)
    print("Ethernet Interface (eth0):")
    print(res.stdout.strip())
except Exception as e:
    print("Error querying ip:", e)



In [ ]:
print("==================================================")
print(" 4. ZYNQ XADC SENSORS (TEMPERATURE & VOLTAGES)")
print("==================================================")
# Look for XADC in Linux IIO subsystem
iio_devices = glob.glob("/sys/bus/iio/devices/iio:device*")
if iio_devices:
    dev = iio_devices[0]
    print(f"XADC Device found: {dev}")
    
    # Read Temperature: (raw + offset) * scale / 1000 = deg C
    try:
        with open(f"{dev}/in_temp0_raw", "r") as f: raw = float(f.read().strip())
        with open(f"{dev}/in_temp0_offset", "r") as f: offset = float(f.read().strip())
        with open(f"{dev}/in_temp0_scale", "r") as f: scale = float(f.read().strip())
        temp_c = (raw + offset) * scale / 1000.0
        print(f"  Zynq Chip Temperature: {temp_c:.1f} °C")
    except Exception as e:
        print("  Temperature sensor unavailable:", e)

    # Read VCCINT (Internal Core Voltage, nominal 1.0V)
    try:
        with open(f"{dev}/in_voltage0_vccint_raw", "r") as f: raw = float(f.read().strip())
        with open(f"{dev}/in_voltage0_vccint_scale", "r") as f: scale = float(f.read().strip())
        vccint = (raw * scale) / 1000.0
        print(f"  VCCINT Core Voltage:   {vccint:.3f} V")
    except Exception:
        pass

    # Read VCCAUX (Auxiliary Voltage, nominal 1.8V)
    try:
        with open(f"{dev}/in_voltage1_vccaux_raw", "r") as f: raw = float(f.read().strip())
        with open(f"{dev}/in_voltage1_vccaux_scale", "r") as f: scale = float(f.read().strip())
        vccaux = (raw * scale) / 1000.0
        print(f"  VCCAUX Aux Voltage:    {vccaux:.3f} V")
    except Exception:
        pass
else:
    print("XADC IIO driver not loaded or not in default sysfs path.")



In [ ]:
print("==================================================")
print(" 5. SD CARD STORAGE UTILIZATION")
print("==================================================")
subprocess.run(["df", "-h", "/", "/boot"], check=False)

